# M2.1_五点格式稀疏矩阵与MMS — 交互式实验工作台（由 HTML 演示网页生成）

> **所属模块**：`02_M2_手写离散_FDM-FVM`  
> **实验定位**：工科本科生计算物理与 SciML 交互式上机实验  
> **运行环境**：本地 Jupyter / VS Code / Colab / 个人笔记本 CPU  

---

## 本讲演示页序导览（Notebook 章节与之严格一一对应）

1. M2.1 五点格式、稀疏矩阵与 MMS
2. ① 工程场景与物理直觉：芯片封装上的稳态温度场
3. ② 失效先行：结果看似合理，矩阵已经坏了
4. ③ 从连续方程到五点模板
5. ④ kron 构造稀疏系统
6. ⑤ 现场运行：解、残差和误差
7. ⑥ 验证三件套
8. ⑦ 机制：一阶 Neumann 为什么把阶拖到 1
9. ⑧ POE：改动前先预测
10. ⑨ 现场诊断挑战：收敛与正确是两本账
11. ⑨ +1 诊断答案与机理解析
12. ⑩ 本节验收清单
13. 总结与下一站

> 对齐说明：本 Notebook 由 `slides/*_演示网页.html` 自动生成；
> 代码一律从 `code/solution.py` 真实导入，函数签名由磁盘源码决定，不会漂移。


## 逐页理论与实验（与演示页序一致）

### 第 2 页 · ① 工程场景与物理直觉：芯片封装上的稳态温度场

- 芯片封装或 PCB 的稳态温度分布，是二维 Poisson 方程最直接的工程原型：热源在芯片位置注入功率，基板与外壳给出恒温或换热边界，工程师真正关心的是 最热点的位置 以及它随散热条件变化的定量关系。
- 离散之后，每个网格节点只与上下左右四个邻居发生能量交换——这既是"五点格式"的物理来源，也解释了为什么这个矩阵 极度稀疏 ：未知数是  $N^2$  量级，非零元约  $5N^2$  量级（内点口径；把边界单位行也算进来的全节点版约  $4.5N^2$ —— $N=41$  实测 7609  $=4.53N^2$ ，两种装配的对照见 ④）。
- 30 秒思考 ：总节点网格从  $41\times41$  加密到  $161\times161$ ，全节点数增加  $(161/41)^2=15.42$  倍；内部自由度从  $39^2$  增至  $159^2$ ，增加  16.62 倍 。交付的全节点矩阵保留边界单位行，实际 nnz 从 7609 增至 126409，增加  16.61 倍 。所以“未知数倍率”须说明指全节点还是消去边界后的内部 DOF；nnz 是装配实测量，不应简单等同未知数倍率。
- <small>口径提醒：网格边长倍率为  $161/41=3.93$ ，因此全节点未知数倍率为  $15.42$ ；消去边界后的内点 DOF 倍率为  $16.62$ ；交付全节点矩阵的 nnz 实测倍率为  $126409/7609=16.61$ 。另一课堂序列  $15/31/63$  有两次加密，端点间距比  $h_{63}/h_{15}=14/62$ ，不能把整个区间描述成只减半。若要估算网格间距减半一次，二维未知数/nnz 量级约增 4 倍，精确比值还取决于边界口径与网格取点数。</small>


### 第 3 页 · ② 失效先行：结果看似合理，矩阵已经坏了

- 一个常见错误是把 Dirichlet 值写入边界行，却没有同步清理边界列。这样做得到的解可能仍然平滑，但矩阵失去对称性，CG（共轭梯度法，一种要求矩阵对称正定的迭代解法）的理论前提不再满足；另一种错误是用一阶单边 Neumann，整体收敛阶从 2 降为 1。
- text
- 正确：A = A.T，残差下降，p ≈ 2
- 错误：max|A-A.T| > 0，CG 异常，或 p ≈ 1
- F4 失效卡 ：先检查矩阵形状、非零元数量、对称性、边界残差和收敛斜率，再决定是否更换求解器。
- 实测证据 （两条失效都有实跑留痕，可复现）： diagnostics/M2.1_F4_cards.md ——卡一「边界行清零未恢复对角」令进程退出码 0 却输出全 NaN；卡二「只划行不清列」实测 nnz 7609/7765、 $\max|A-A^T|=1.0$ ；卡三「一阶 Neumann」实测  $p=1.017\to1.004$ 。复现命令见卡内附录。


### 第 4 页 · ③ 从连续方程到五点模板

- 对均匀网格  $x_i=i h$ 、 $y_j=j h$ ，中心差分为
- $-\nabla_h^2u_{ij}=\frac{4u_{ij}-u_{i+1,j}-u_{i-1,j}-u_{i,j+1}-u_{i,j-1}}{h^2}.$
- 模板中心系数为  $4/h^2$ ，四个邻居为  $-1/h^2$ 。边界节点不能用内部模板代替，必须按 Dirichlet、Neumann 或 Robin 的物理约束重新写行。
- 制造解选择
- $u^*(x,y)=\sin(\pi x)\sin(\pi y),\quad f^*=2\pi^2u^*,$
- 边界值恰为零，便于把内部误差和边界误差分开。


### 第 5 页 · ④ kron 构造稀疏系统

- python · `kron` 构造稀疏系统
- 可编辑 · Shift+Enter 运行  ↺ 重置  ▶ 运行
- import  numpy  as  np
- from  scipy.sparse  import  diags, eye, kron
- from  scipy.sparse.linalg  import  spsolve
- def  poisson_matrix(n):


### 第 6 页 · ⑤ 现场运行：解、残差和误差

- python · 现场运行：解、残差和误差
- 可编辑 · Shift+Enter 运行  ↺ 重置  ▶ 运行
- for  n  in  ( 15 ,  31 ,  63 ):
- A, h = poisson_matrix(n)
- X, Y, b_grid, u_ex = manufactured_rhs(n)
- u = spsolve(A, b_grid.ravel()).reshape(n, n)


### 第 7 页 · ⑥ 验证三件套

- 验证项
- 实施方法
- 通过标准
- 精确解  与  $u^*$  比较  $L^2,L^\infty$   误差有限且边界满足
- 收敛阶   $p=\log_2(e_h/e_{h/2})$   五点内点格式接近 2
- 方程残差   $\|Au-b\|_2$   与线性求解容差一致


### 第 8 页 · ⑦ 机制：一阶 Neumann 为什么把阶拖到 1

- 验证三件套只对「五点内点格式」承诺接近 2；换成 一阶单边 Neumann ，理论阶数改变，必须另列一行说明（正文⑥注）。机制如下：
- 单边一阶差分  $(u_{N-1}-u_{N-2})/h = g$  的泰勒展开残留缺陷  $-\frac{h\,u''(1)}{2}$ ，落在系数为  $O(1)$  的 边界行 上；经离散 Green 函数放大，成为  $O(h)$  的 全局 误差——整体阶数因此从 2 掉到 1。二阶单边  $(3u_{N-1}-4u_{N-2}+u_{N-3})/(2h)=g$  的缺陷是  $O(h^3)$ ，全局仍为  $O(h^2)$ 。
- 边界处理（一维最小例  $-u''=f$ ， $u^*=e^x$ ， $u''(1)=e\neq0$ ）
- 21→41
- 41→81
- 81→161


### 第 9 页 · ⑧ POE：改动前先预测

- 💡  POE 自检台 ：先独立思考（可点击填写预测），再点击右侧按钮揭晓标准解析对照。
- 👁️ 全部揭晓 / 隐藏
- 运行前预测
- 观测证据
- 解释/失效归类
- 自检操作


### 第 10 页 · ⑨ 现场诊断挑战：收敛与正确是两本账

- 现场背景： 故障一 · 残差极小，阶却只有 1 ——把右边界换成一阶单边 Neumann 后重跑 MMS：代数残差依旧  residual≈1e-14 （求解器侧无可挑剔——CSV 实测 N=41 为 1.13e-14），但收敛阶从 2 掉到 1：每加密一倍，误差只减半，而不是降到四分之一。 故障二 · 解很平滑，CG 却不肯收敛 ——二维 Poisson 稀疏求解：把 Dirichlet 边界值写入边界行后直接调用 CG，残差长期震荡不下降；换成直接法却能解出看起来合理的解。请诊断核心病因：
- 课程立场 ：AI 可以帮你生成  kron  代码和收敛表，但不能替你判断哪一项输出是离散误差、线性求解误差还是后处理索引错误。


### 第 11 页 · ⑨ +1 诊断答案与机理解析

- 诊断题一 · 残差 vs 收敛阶（F4） ：标准答案  C ——一阶边界离散或制造解/网格定义不一致。残差度量「离散代数方程被解到多准」，误差度量「离散解与 PDE 真解的距离」，两者隔着截断误差与边界处理；残差  $1.13\text{e-}14$  而阶只有 1，最典型是一阶单边 Neumann 把整体收敛阶拖到 1（卡三实测  $p=1.017\to1.004$ ）。误区：A 收紧求解容差对不降的截断误差无效；B float64 舍入是  $10^{-16}$  量级；D 网格循环失效则误差完全不变、阶也算不出来。
- 诊断题二 · 装配对称性（F4） ：标准答案  C ——边界列未对称清理， $A\ne A^T$ ，CG 的对称正定前提被破坏；先查  $\max|A-A^T|$ （卡二实测  $=1.0$ ）再谈换求解器。直接法解差  $6.2\times10^{-15}$  说明「解看起来对」不能证明装配对。误区：A 增加迭代次数不改变被破坏的前提；B 换 GMRES 是用更贵的方法掩盖装配 bug；D 条件数影响收敛速度，不会让 CG 完全失效。
- 课程立场 ：AI 可以帮你生成  kron  代码和收敛表，但不能替你判断哪一项输出是离散误差、线性求解误差还是后处理索引错误。


### 第 12 页 · ⑩ 本节验收清单

- 能从五点模板写出稀疏矩阵非零结构；
- 能使用 MMS 计算精确解误差和收敛阶；
- 能检查边界处理对称性与 Neumann 阶数；
- 能解释  kron 、展平顺序和图像坐标之间的关系；
- pytest -q  通过全部测试（含两条失效模式回归：非对称装配、Neumann 掉阶）；
- 将输出和元数据保存到  benchmarks/poisson_mms/ 。


### 第 13 页 · 总结与下一站

- 五点格式的价值不在于公式短，而在于每个系数、边界行和误差斜率都能追溯。下一节  M2.2 热传导时间推进与稳定性实验  将把空间离散与时间步结合，展示扩散数超过阈值后舍入误差如何被放大成 F2 爆炸。


## 演示页代码实验台
下面单元格直接抽取自演示网页的可运行代码（`cell-src`），按顺序执行即可复现网页上的交互结果。

In [ ]:

import numpy as np
from scipy.sparse import diags, eye, kron
from scipy.sparse.linalg import spsolve

def poisson_matrix(n):
    h = 1.0/(n+1)
    T = diags([-np.ones(n-1), 2*np.ones(n), -np.ones(n-1)], [-1,0,1])
    A = (kron(eye(n), T) + kron(T, eye(n))) / h**2
    return A.tocsr(), h

def manufactured_rhs(n):
    h = 1.0/(n+1)
    x = np.arange(1, n+1)*h
    X, Y = np.meshgrid(x, x, indexing="ij")
    u_exact = np.sin(np.pi*X)*np.sin(np.pi*Y)
    return X, Y, 2*np.pi**2*u_exact, u_exact

# 双装配口径实测：kron 内点版 vs 全节点显式边界版（n=9）
def assemble_full_node(n):
    h = 1.0/(n+1)
    N = n + 2
    A = np.zeros((N*N, N*N))
    for i in range(N):
        for j in range(N):
            k = i*N + j
            if 0 < i < N-1 and 0 < j < N-1:
                A[k, k] = 4.0/h**2
                A[k, k-1] = A[k, k+1] = -1.0/h**2
                A[k, k-N] = A[k, k+N] = -1.0/h**2
            else:
                A[k, k] = 1.0
    return A, N

n = 9
A_in, h = poisson_matrix(n)
A_full, N = assemble_full_node(n)
idx = [i*N + j for i in range(1, N-1) for j in range(1, N-1)]
A_blk = A_full[np.ix_(idx, idx)]
print(f"n={n}: 内点块逐元素一致 allclose={np.allclose(A_in.toarray(), A_blk)}, "
      f"max|Delta|={np.abs(A_in.toarray()-A_blk).max():.1e}, "
      f"nnz {A_in.nnz} = {(A_blk!=0).sum()}")
  

In [ ]:

for n in (15, 31, 63):
    A, h = poisson_matrix(n)
    X, Y, b_grid, u_ex = manufactured_rhs(n)
    u = spsolve(A, b_grid.ravel()).reshape(n, n)
    err_l2 = np.sqrt(h*h*np.sum((u-u_ex)**2))
    err_inf = np.max(np.abs(u-u_ex))
    residual = np.linalg.norm(A @ u.ravel() - b_grid.ravel())
    print(f"n={n:2d}, h={h:.4e}, L2={err_l2:.4e}, Linf={err_inf:.4e}, residual={residual:.2e}")
  

### 源码：`env_check.py`

In [ ]:

# -*- coding: utf-8 -*-
"""M2.1 环境与算法自检：五点格式稀疏矩阵 + 制造解收敛（网页运行与「下载」同源）。

用法：
    python code/env_check.py          # 本地终端直接跑
    网页「运行」按钮                    # Pyodide 内运行同一份源码

自检三件套（对应演示页⑥「验证三件套」）：
    1. 解：与制造解 u* = sin(pi x) sin(pi y) 逐点对照
    2. 残差：‖A u - b‖（直接法应降到机器精度量级）
    3. 误差：真 L2 口径 sqrt(h^2 * sum(e^2)) 与 Linf，并给出收敛阶

口径备注（2026-10-16 统一）：
    * 本脚本与源稿④/⑤、CSV、tests/test_mms.py 一致，用「真 L2」范数；
      含端点 N×N 网格上真 L2/RMS=N/(N-1)=1+1/(N-1)，这个网格因子把 RMS 阶压低约 0.03，
      历史数字 1.97-1.99 即来源于此，双向留档于 diagnostics/M2.1_F4_cards.md。
    * 矩阵用 kron 内点版（5N^2 非零元、除 h^2）；与 code/solution.py 的
      全节点版 assemble_poisson_system 在内点块上逐元素一致，差异仅在边界行
      形式与 h^2 放缩（见源稿④「双装配口径」长注）。
"""
import numpy as np
from scipy.sparse import diags, eye, kron
from scipy.sparse.linalg import spsolve


def poisson_matrix(n):
    """内点五点格式：kron 构造 n^2 x n^2，除 h^2（源稿④口径）。"""
    h = 1.0 / (n + 1)
    T = diags([-np.ones(n - 1), 2 * np.ones(n), -np.ones(n - 1)], [-1, 0, 1])
    A = (kron(eye(n), T) + kron(T, eye(n))) / h ** 2
    return A.tocsr(), h


def manufactured_rhs(n):
    """制造解 u* = sin(pi x) sin(pi y)，右端 f = 2 pi^2 u*。"""
    h = 1.0 / (n + 1)
    x = np.arange(1, n + 1) * h
    X, Y = np.meshgrid(x, x, indexing="ij")
    u_exact = np.sin(np.pi * X) * np.sin(np.pi * Y)
    return X, Y, 2 * np.pi ** 2 * u_exact, u_exact


def main():
    print("=" * 66)
    print("M2.1 自检：五点格式稀疏矩阵 + 制造解（MMS）收敛")
    print("=" * 66)
    prev = None
    for n in (15, 31, 63):
        A, h = poisson_matrix(n)
        X, Y, b_grid, u_ex = manufactured_rhs(n)
        u = spsolve(A, b_grid.ravel()).reshape(n, n)
        err_l2 = float(np.sqrt(h * h * np.sum((u - u_ex) ** 2)))
        err_inf = float(np.max(np.abs(u - u_ex)))
        residual = float(np.linalg.norm(A @ u.ravel() - b_grid.ravel()))
        order = float("nan") if prev is None else float(np.log2(prev / err_l2))
        print("n=%2d, h=%.4e, L2=%.4e, Linf=%.4e, residual=%.2e, p=%.3f"
              % (n, h, err_l2, err_inf, residual, order))
        prev = err_l2
    # 对称性体检：CG 类迭代法要求对称正定；这里顺带给出判据
    A31, _ = poisson_matrix(31)
    print("max|A-A^T| = %.1e（kron 内点版恒为 0；只划行不清列会变成 1.0，见 F4 卡二）"
          % float(np.abs(A31 - A31.T).max()))


if __name__ == "__main__":
    main()

## 加载本单元真实源码并核对函数签名
下面直接导入 `code/solution.py`（**不是副本**）；若你在磁盘上改了代码，这里的签名会立刻跟着变。

In [ ]:
# --- 先定位本单元目录：从 notebooks/ 或项目根目录启动 Jupyter 都能找到 code/solution.py
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
code_candidates = [parent / "code" for parent in search_roots]
for root in search_roots:
    code_candidates.extend(root.glob("*/*/code"))
code_candidates = list(dict.fromkeys(p.resolve() for p in code_candidates))
found = [p.resolve() for p in code_candidates if (p / "solution.py").is_file()]
# 从项目根目录启动时会有多个单元命中，优先锁定本单元（M2.1），避免误载他单元的 solution.py
preferred = [p for p in found if any("M2.1" in part for part in p.parts)]
CODE_DIR = (preferred or found or [None])[0]
if CODE_DIR is None:
    raise FileNotFoundError("找不到 code/solution.py，请从本单元目录或项目根目录启动 Jupyter")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
ASSET_DIR = CODE_DIR.parent / "assets"
print("CODE_DIR  =", CODE_DIR)
print("ASSET_DIR =", ASSET_DIR)


In [ ]:
import sys, inspect, importlib.util, os
SOL_PATH = os.path.join(CODE_DIR, "solution.py")
MODULE_NAME = "course_unit_M2_1_solution"
spec = importlib.util.spec_from_file_location(MODULE_NAME, SOL_PATH)
S = importlib.util.module_from_spec(spec)
# Make paths relative to the owning unit so main() is independent of Jupyter's cwd.
_OLD_CWD = os.getcwd()
try:
    os.chdir(CODE_DIR.parent)
    spec.loader.exec_module(S)
finally:
    os.chdir(_OLD_CWD)
sys.modules[MODULE_NAME] = S
print("已从磁盘加载:", SOL_PATH)
print()
def _show():
    import ast
    tree = ast.parse(open(SOL_PATH, encoding="utf-8").read())
    for n in tree.body:
        if isinstance(n, ast.FunctionDef) and not n.name.startswith("_"):
            print("def %s" % n.name, inspect.signature(getattr(S, n.name)))
        elif isinstance(n, ast.ClassDef):
            print("class", n.name)
            for m in n.body:
                if isinstance(m, ast.FunctionDef) and (not m.name.startswith("_") or m.name == "__init__"):
                    print("    ", m.name, [a.arg for a in m.args.args])
_show()

## 运行主实验
执行下方单元格即调用本单元真实入口 `main()`，输出与演示网页一致。

In [ ]:
# 执行本单元真实入口函数（网页上的数字由此产生）
try:
    S.main()
except Exception as e:
    print("运行未成功，请把完整报错发给老师：")
    import traceback; traceback.print_exc()

## 物理场成果图展示（相对路径，换机可用）

In [ ]:
from IPython.display import Image, display
import glob
pngs = sorted(glob.glob(os.path.join(ASSET_DIR, "*.png")))
if not pngs:
    print("assets 目录下暂无 png；请先运行上面的求解入口生成图表。")
for f in pngs:
    print("展示:", os.path.basename(f))
    display(Image(filename=f))

## 4. 工科思考与参数敏感度探索 (Exploration Challenges)

### 诊断与思考题：
1. **稀疏矩阵存储的生死攸关性**：将网格尺寸从 $30\times 30$ 加密到 $100\times 100$。计算此时稀疏刚度矩阵的非零元数量占比（从 $\sim 0.5\%$ 暴跌至 $\sim 0.05\%$）。为什么严禁用稠密二维数组 `np.zeros((N*N, N*N))` 存储工程差分矩阵？（后者在 $N=200$ 时就会瞬间耗尽 32GB 内存引发内存溢出！）。
2. **制造解法 (MMS) 收敛阶验证实操**：修改代码中的网格密度序列为 $N = [15, 30, 60, 120]$，运行求解并记录对应的相对 $L_2$ 误差。在双对数坐标系下拟合直线斜率，严格验证手写五点格式是否达到二阶理论收敛精度 $\mathcal{O}(h^2)$。
3. **Neumann 绝热边界的二阶保阶对策**：如果矩形板右侧改为绝热边界 $\partial u/\partial x = 0$。如果直接采用单边一阶向前差分 $(u_N - u_{N-1})/h = 0$，为什么会导致全域本来二阶的有限差分精度被拖拽退化为一阶？如何通过引入虚节点 (Ghost Node) 保证边界处仍然具备二阶中心差分精度？